# Flow compensation in a 2D gradient echo

A spoiled gradient echo excites a slice, encodes one line of k-space, and reads it out while the
readout gradient sweeps through the echo. The prephaser before the readout cancels the readout
lobe's area up to that instant, which puts $k = 0$ there — so a **stationary** spin arrives at the
echo with no phase from the readout axis. It is the workhorse acquisition for fast T1-weighted and
angiographic imaging.

That cancellation is a statement about the **zeroth** gradient moment only. The phase a spin
accumulates along one axis is

$$\phi = 2\pi \int G(t)\, x(t)\, \mathrm{d}t = 2\pi\left(m_0 x_0 + m_1 v\right),
\qquad m_n = \int G(t)\, (t - t_\text{echo})^n\, \mathrm{d}t$$

where $x_0$ is where the spin started and $v$ its velocity along that axis. Setting $m_0 = 0$
removes the first term. It says nothing about $m_1$, which is not zero, so a spin **moving** along
the axis arrives with a phase proportional to its velocity.

Flowing blood, CSF and a moving vessel wall all carry that term, and because it varies from shot
to shot with the flow it appears in the image as ghosting along the phase-encode direction and as
signal loss where the velocity varies inside a voxel.

**Flow compensation**, or gradient moment nulling, removes it by reshaping the gradients so that
$m_1 = 0$ at the echo as well as $m_0$. On the readout axis the canonical construction uses three
lobes before the echo rather than two:

```text
ordinary        [ winder ] [ readout ->  echo         m0 = 0, m1 != 0
compensated  [ w1 ][ w2 ] [ readout ->  echo          m0 = 0, m1 = 0
```

This notebook builds both, measures the moments on the emitted waveform, and shows what the
compensation costs in echo time.

| | |
|---|---|
| **1** | the first moment of an ordinary gradient echo |
| **2** | asking for flow compensation |
| **3** | measuring it on the emitted repetition |
| **4** | what it costs, and on which axis |
| **5** | the files |

**Output:** two `.seq` files, an ordinary gradient echo and a flow-compensated one.
**Needs nothing but `seqcraft`.**

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pypulseq as pp

import seqcraft as sc

opts = pp.Opts(
    max_grad=40, grad_unit='mT/m',
    max_slew=150, slew_unit='T/m/s',
    B0=3.0,
    rf_dead_time=100e-6,
    rf_ringdown_time=30e-6,
    adc_dead_time=10e-6,
)

FOV_MM, MATRIX, THICKNESS_MM = 220.0, 64, 5.0
BANDWIDTH_HZ_PX = 500.0
SEQ_DIR = Path('seq')
SEQ_DIR.mkdir(exist_ok=True)

---

## 1. The first moment of an ordinary gradient echo

`GRE2DTR` is one repetition: excitation, phase encode, readout, spoiler. Its moments are a
property of the waveform it emits, so the way to read them is to integrate the events themselves
between the excitation and the echo.

In [ ]:
def moment(shot, order, axis, kernel):
    """The order-th gradient moment on one axis, from the excitation to the echo."""
    origin, echo = kernel.exc.time_to_center(), kernel.time_to_echo()
    total = 0.0
    for start, event, _ in sc.flatten(shot):
        if getattr(event, 'type', None) not in ('trap', 'grad'):
            continue
        if getattr(event, 'channel', None) != axis:
            continue
        times, amps = sc.events.knots_of(event, start)
        if times.size < 2 or echo <= times[0]:
            continue
        if echo < times[-1]:                       # the readout lobe straddles the echo
            cut = int(np.searchsorted(times, echo))
            edge = float(np.interp(echo, times, amps))
            times, amps = np.append(times[:cut], echo), np.append(amps[:cut], edge)
        total += sc.events.pwl_moment(times - origin, amps, order)
    return float(total)


ordinary = sc.modules.GRE2DTR(opts=opts, fov_mm=FOV_MM, matrix=(MATRIX, MATRIX),
                              thickness_mm=THICKNESS_MM, flip_deg=20.0,
                              bandwidth_hz_px=BANDWIDTH_HZ_PX)
shot = ordinary(line=MATRIX // 2)

print(f'an ordinary gradient echo, TE = {ordinary.te_s * 1e3:.3f} ms\n')
print(f'   m0 on x at the echo   {moment(shot, 0, "x", ordinary):+12.3e}  1/m')
print(f'   m1 on x at the echo   {moment(shot, 1, "x", ordinary):+12.3e}  s/m')

v = 0.5
turns = moment(shot, 1, 'x', ordinary) * v
print(f'\na spin moving at {v} m/s along x therefore arrives with {turns:+.2f} turns '
      f'({turns * 2:+.2f} pi) of phase')

`m0` is at the arithmetic floor — that is what the prephaser is for, and it is why a stationary
spin is unaffected. `m1` is not, and half a metre per second is an ordinary velocity in the aorta
or the carotid.

The phase itself is not the problem; a constant phase would be invisible in magnitude. The problem
is that it is proportional to velocity, so it differs between shots as the flow pulses, and a
difference between shots along the phase-encode direction is a ghost.

---

## 2. Asking for flow compensation

The requirement is physical — *no first moment at the echo, on this axis* — so that is what you
say. Which part of the repetition has to change to deliver it is the repetition's problem.

In [ ]:
compensated = sc.modules.GRE2DTR(opts=opts, fov_mm=FOV_MM, matrix=(MATRIX, MATRIX),
                                 thickness_mm=THICKNESS_MM, flip_deg=20.0,
                                 bandwidth_hz_px=BANDWIDTH_HZ_PX,
                                 flow_comp=sc.FlowCompensation(axis='x'))

print(f'{"":>14}{"prephaser lobes":>18}{"areas / (1/m)":>30}{"total":>12}')
for name, kernel in (('ordinary', ordinary), ('compensated', compensated)):
    areas = [float(lobe.area) for lobe in kernel.ro.prephaser_lobes]
    written = '  '.join(f'{a:+9.2f}' for a in areas)
    print(f'{name:>14}{len(areas):18d}{written:>30}{sum(areas):+12.2f}')

The compensated winder is two lobes of opposite sign rather than one, and their areas still sum to
the same total — the echo condition is not traded away to get the first moment. Two lobes give two
degrees of freedom, and two moments to null need exactly two.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(9.0, 4.0), sharex=True, sharey=True)
for ax, (name, kernel) in zip(axes, (('ordinary', ordinary), ('compensated', compensated))):
    for start, event, _ in sc.flatten(kernel(line=MATRIX // 2)):
        if getattr(event, 'channel', None) != 'x':
            continue
        times, amps = sc.events.knots_of(event, start)
        ax.plot(times * 1e3, np.asarray(amps) / 1e3, color='#2b6cb0')
    ax.axvline(kernel.time_to_echo() * 1e3, color='#c05621', ls='--', lw=1.0)
    ax.axhline(0.0, color='0.8', lw=0.8)
    ax.set_ylabel('G$_x$ / (kHz/m)')
    ax.set_title(name, loc='left', fontsize=10)
axes[-1].set_xlabel('time / ms')
fig.suptitle('readout axis, excitation to echo (dashed)', fontsize=10)
fig.tight_layout()
plt.show()

---

## 3. Measuring it on the emitted repetition

The moments below are integrated over every gradient the repetition plays on that axis, not over
the prephaser alone, and across three protocols with different readout durations.

In [ ]:
print(f'{"bandwidth":>12}{"matrix":>8}{"moment":>12}{"ordinary":>14}{"compensated":>14}')
for bw, n in ((300.0, 64), (500.0, 64), (900.0, 128)):
    made = {}
    for name, extra in (('ordinary', {}),
                        ('compensated', {'flow_comp': sc.FlowCompensation(axis='x')})):
        made[name] = sc.modules.GRE2DTR(opts=opts, fov_mm=FOV_MM, matrix=(n, n),
                                        thickness_mm=THICKNESS_MM, flip_deg=20.0,
                                        bandwidth_hz_px=bw, **extra)
    for order, unit in ((0, '1/m'), (1, 's/m')):
        values = {k: moment(v(line=n // 2), order, 'x', v) for k, v in made.items()}
        print(f'{bw:12.0f}{n:8d}{f"m{order} / {unit}":>12}'
              f'{values["ordinary"]:+14.3e}{values["compensated"]:+14.3e}')

`m0` stays at the floor in both columns across every protocol — the echo is still where it should
be — and `m1` drops from something a moving spin can feel to the same floor.

---

## 4. What it costs, and on which axis

Three lobes before the echo take longer than two, so the echo moves later.

In [ ]:
print(f'{"":>14}{"winder / ms":>14}{"TE / ms":>10}{"TR / ms":>10}')
for name, kernel in (('ordinary', ordinary), ('compensated', compensated)):
    print(f'{name:>14}{kernel.winder_s * 1e3:14.3f}{kernel.te_s * 1e3:10.3f}'
          f'{kernel.tr_s * 1e3:10.3f}')
print(f'\nthe echo moves out by {(compensated.te_s - ordinary.te_s) * 1e3:.3f} ms')

A later echo means more T2\* decay and more time for off-resonance to accumulate, which is the
trade every flow-compensated protocol makes.

Flow compensation is per axis, and a repetition can carry it on more than one. Which axes are
available depends on the sequence: this one owns the phase-encode winder and the readout
prephaser, while its z gradient is the slice rephaser that the excitation realises for itself.

In [ ]:
for axis in ('x', 'y', ('x', 'y'), 'z'):
    try:
        kernel = sc.modules.GRE2DTR(opts=opts, fov_mm=FOV_MM, matrix=(MATRIX, MATRIX),
                                    thickness_mm=THICKNESS_MM, flip_deg=20.0,
                                    bandwidth_hz_px=BANDWIDTH_HZ_PX,
                                    flow_comp=sc.FlowCompensation(axis=axis))
    except sc.ConfigurationError as refused:
        print(f'axis={axis!r:12} refused: {str(refused).splitlines()[0]}')
        continue
    named = (axis,) if isinstance(axis, str) else axis
    shot = kernel(line=MATRIX // 4)
    got = '  '.join(f'm1({a}) = {moment(shot, 1, a, kernel):+9.2e}' for a in named)
    print(f'axis={axis!r:12} TE {kernel.te_s * 1e3:6.3f} ms   {got}')

Compensating x and y together costs more echo time than either alone, because both winders play in
the same interval and the longer of the two sets it.

A 3D sequence answers differently: `GRE3DTR` owns z as well, because there the z gradient carries
the partition encoding and the slab rephasing together.

---

## 5. The files

In [ ]:
print(f'{"":>14}{"blocks":>9}{"duration / s":>14}   file')
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always', sc.SeqCraftWarning)
    for name, kernel in (('ordinary', ordinary), ('compensated', compensated)):
        scan = sc.LogicBlock('flowcomp_gre_2d')
        for line in range(MATRIX):
            scan.add(line * kernel.tr_s, kernel(line=line, phase_deg=117.0 * line))
        seq = sc.compile(scan, opts)
        path = SEQ_DIR / f'flowcomp_gre_2d_{name}.seq'
        seq.write(str(path))
        print(f'{name:>14}{len(seq.block_events):9d}{seq.duration()[0]:14.3f}   {path}')

for warning in caught:
    print(f'\n{warning.message}')

---

## Summary

A gradient echo cancels the readout gradient's zeroth moment at the echo, which puts $k = 0$
there and leaves a stationary spin unaffected. The first moment is not zero, so a spin moving
along that axis arrives with a phase $2\pi m_1 v$ — and because the flow pulses, that phase differs
between shots and appears as ghosting and signal loss.

Flow compensation nulls $m_1$ as well, by splitting the prephaser into two lobes of opposite sign
whose areas still sum to the same total. Measured on the emitted repetition across three
protocols, $m_1$ on the readout axis falls to the arithmetic floor while $m_0$ stays there. The
cost is a later echo, because three lobes before it take longer than two.

**What this notebook does not cover.** Compensation is applied at the first echo; later echoes of a
multi-echo train accumulate their own first moment from the lobes between them. Nulling the second
moment — acceleration — needs a further lobe and is not implemented. And flow compensation removes
the velocity phase rather than measuring it: to *measure* velocity you deliberately create a known
first-moment difference between two acquisitions, which is phase contrast, in
[`pc_gre_2d/`](../pc_gre_2d/).

## References

Bernstein, King and Zhou, *Handbook of MRI Pulse Sequences*, Elsevier 2004 — §9.2 covers gradient
moment nulling, including the two-lobe construction used here and the extension to higher moments.

Pattany et al., *Motion artifact suppression technique (MAST) for MR imaging*, J Comput Assist
Tomogr 11(3):369-377, 1987 — the original description of first-moment nulling for artefact
suppression.